### Heart Failure

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the dataset
url = 'https://raw.githubusercontent.com/sauravmishra1710/Heart-Failure-Condition-And-Survival-Analysis/master/Data/heart_failure_clinical_records_dataset.csv'
heart_data = pd.read_csv(url)

# Define features (X) and target variable (y)
X = heart_data.drop(columns=['DEATH_EVENT'])  # Features
y = heart_data['DEATH_EVENT']  # Target variable

#with smote
from imblearn.over_sampling import SMOTE
# Apply SMOTE
smote = SMOTE(random_state=42)
X, y = smote.fit_resample(X, y)
X_resampled = X
X_resampled = y
# Display class distribution after SMOTE
print("\nClass Distribution After SMOTE:")
print(y.value_counts())
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Class Distribution After SMOTE:
DEATH_EVENT
1    203
0    203
Name: count, dtype: int64


###Scaling

In [ ]:
# prompt: scale the data

from sklearn.preprocessing import MinMaxScaler

# Create a MinMaxScaler object
scaler = MinMaxScaler()

# Fit the scaler to the training data
scaler.fit(X_train)

# Transform the training and test data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = X_train_scaled
X_test = X_test_scaled

###KNN - Manual Search

In [ ]:
# prompt: apply knn manual search

from sklearn.neighbors import KNeighborsClassifier

# Initialize lists to store scores
train_score = []
test_score = []

# Define the range of k values to be tested
k_values = range(1, 20)

# Loop through different values of k
for k in k_values:
  # Initialize and fit the KNN classifier
  knn = KNeighborsClassifier(n_neighbors=k)
  knn.fit(X_train, y_train)

  # Calculate and store the scores
  train_score.append(knn.score(X_train, y_train))
  test_score.append(knn.score(X_test, y_test))

# Find the k value with the highest test score
best_k = test_score.index(max(test_score)) + 1

# Print the best k and its corresponding scores
print(f"Best k: {best_k}")
print(f"Train score with k = {best_k}: {train_score[best_k - 1]}")
print(f"Test score with k = {best_k}: {test_score[best_k - 1]}")


Best k: 4
Train score with k = 4: 0.8395061728395061
Test score with k = 4: 0.8048780487804879


###KNN-Grid Search

In [ ]:
# prompt: apply knn grid search

from sklearn.model_selection import GridSearchCV

# Define the grid of hyperparameters to search
param_grid = {
    'n_neighbors': range(1, 20)
    # 'weights': ['uniform', 'distance'],
    # 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Create a KNN classifier
knn = KNeighborsClassifier()

# Initialize the GridSearchCV object
grid_search = GridSearchCV(knn, param_grid, cv=5)

# Fit the grid search object to the training data
grid_search.fit(X_train, y_train)

# Get the best hyperparameters
best_params = grid_search.best_params_

# Print the best hyperparameters
print("Best hyperparameters:")
print(best_params)

# Get the best KNN model
best_knn = grid_search.best_estimator_

# Evaluate the best KNN model on the test data
test_score = best_knn.score(X_test, y_test)

# Print the test score
print(f"Test score with best hyperparameters: {test_score}")


Best hyperparameters:
{'n_neighbors': 1}
Test score with best hyperparameters: 0.7073170731707317


###KNN-Random Search

In [ ]:
# prompt: apply knn with random search

from sklearn.model_selection import RandomizedSearchCV

# Define the grid of hyperparameters to search
param_grid = {
    'n_neighbors': range(1, 20)
    # 'weights': ['uniform', 'distance'],
    # 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Create a KNN classifier
knn = KNeighborsClassifier()

# Initialize the RandomizedSearchCV object
random_search = RandomizedSearchCV(knn, param_grid, cv=5, n_iter=10)

# Fit the random search object to the training data
random_search.fit(X_train, y_train)

# Get the best hyperparameters
best_params = random_search.best_params_

# Print the best hyperparameters
print("Best hyperparameters:")
print(best_params)

# Get the best KNN model
best_knn = random_search.best_estimator_

# Evaluate the best KNN model on the test data
test_score = best_knn.score(X_test, y_test)

# Print the test score
print(f"Test score with best hyperparameters: {test_score}")


Best hyperparameters:
{'n_neighbors': 1}
Test score with best hyperparameters: 0.7073170731707317


###KNN Hyperopt

In [ ]:
# prompt: apply KNN Hyperopt

import numpy as np
from sklearn.model_selection import cross_val_score
from hyperopt import hp, tpe, fmin, STATUS_OK, Trials

# Define the hyperparameter search space
space = {
    'n_neighbors': hp.choice('n_neighbors', range(1, 20))
    # 'weights': hp.choice('weights', ['uniform', 'distance']),
    # 'algorithm': hp.choice('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
}

# Define the objective function
def objective(params):
  # Initialize the KNN classifier with the given hyperparameters
  knn = KNeighborsClassifier(**params)

  # Fit the KNN classifier on the training data
  knn.fit(X_train, y_train)

  # Perform cross-validation and calculate the negative accuracy score
  score = -np.mean(cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy'))

  # Return the loss dictionary
  return {'loss': score, 'status': STATUS_OK, 'model': knn}

  # Perform cross-validation and calculate the negative accuracy score
  score = -np.mean(cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy'))

  # Return the loss dictionary
  return {'loss': score, 'status': STATUS_OK, 'model': knn}

# Run the optimization
trials = Trials()
best_params = fmin(fn=objective,
                  space=space,
                  algo=tpe.suggest,
                  max_evals=10,
                  trials=trials)

# Get the best KNN model
best_knn = trials.best_trial['result']['model']

# Evaluate the best KNN model on the test data
test_score = best_knn.score(X_test, y_test)

# Print the best hyperparameters and test score
print("Best hyperparameters:")
print(best_params)
print(f"Test score with best hyperparameters: {test_score}")


100%|██████████| 10/10 [00:01<00:00,  5.84trial/s, best loss: -0.753125]
Best hyperparameters:
{'n_neighbors': np.int64(0)}
Test score with best hyperparameters: 0.7073170731707317


###SVM Manual Search

In [ ]:
#Manual Search CV
# Create and train the SVM classifier
svm_classifier = SVC(kernel='linear', random_state=42)
svm_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = svm_classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(classification_rep)


Accuracy: 0.80
Confusion Matrix:
[[33  8]
 [ 8 33]]
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        41
           1       0.80      0.80      0.80        41

    accuracy                           0.80        82
   macro avg       0.80      0.80      0.80        82
weighted avg       0.80      0.80      0.80        82



In [ ]:
import multiprocessing

cores = multiprocessing.cpu_count() # Count the number of cores in a computer
cores

2

###SVM Grid Search

1 Million people (A,B,C,D)

ABC

Grid Search --- Check every area --- A,B,C,D----B  All permutations and combinations

Random Search --- B & C --- Narrow down our results  #do some iterations and find the best
1. Random SearchCV -- B&C
2. Grid Search - B,C

In [ ]:
# prompt: apply grid search hyperparameter tuning svm c, kernel, gamma

from sklearn.model_selection import GridSearchCV

# Define the hyperparameter grid
grid = {
    'C': [1,10,100,1000],  # Regularization parameter
    'kernel': ['linear',  'rbf'],  # Kernel type
    'gamma': ['scale', 'auto', 0.1, 0.2,0.3,0.5,0.6]  # Kernel coefficient
}

# Create the grid search object
grid_search = GridSearchCV(svm_classifier, param_grid = grid, cv=5, scoring='accuracy',n_jobs = -1)

# Fit the grid search object to the training data
grid_search.fit(X_train, y_train)

# Print the best score
print("Best Hyperparameters:")
print(grid_search.best_score_)


# Print the best hyperparameters
print("Best Hyperparameters:")
print(best_params)
best_params = grid_search.best_params_

Best Hyperparameters:
0.8334134615384616
Best Hyperparameters:
{'n_neighbors': np.int64(0)}


###Randmized CV - SVM

In [ ]:
# prompt: apply randomsearch cv

from sklearn.model_selection import RandomizedSearchCV

# Define the hyperparameter grid
grid = {
    'C': [1,10,100,1000],  # Regularization parameter
    'kernel': ['linear',  'rbf'],  # Kernel type
    'gamma': ['scale', 'auto', 0.1, 0.2,0.3,0.5,0.6]  # Kernel coefficient
}

# Create the random search object
random_search = RandomizedSearchCV(svm_classifier, param_distributions = grid, cv=5,n_jobs = -1, n_iter=10) #scoring='accuracy' by default

# Fit the random search object to the training data
random_search.fit(X_train, y_train)

# Print the best score
print("Best Hyperparameters:")
print(random_search.best_score_)

# Print the best hyperparameters
print("Best Hyperparameters:")
print(random_search.best_params_)


Best Hyperparameters:
0.82125
Best Hyperparameters:
{'kernel': 'rbf', 'gamma': 'scale', 'C': 1}


###Randomized - Pipleline - PCA+RF

In [ ]:
# prompt: apply random search random forest with pipeline pca

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

pca = PCA()
rf = RandomForestClassifier()

# Define the pipeline
pipeline = Pipeline(steps=[
    ('pca', pca),
    ('random_forest', rf)
])

# Define the hyperparameter grid
param_grid = {
    'pca__n_components': np.arange(5,10),
    'rf__n_estimators': np.arange(100,1500,100),
    'rf__max_depth': np.arange(1,20),
    'rf__criterion': ["gini", "entropy"]
}

# Create the random search object
random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, cv=5, n_jobs=-1, n_iter=10)

# Fit the random search object to the training data
random_search.fit(X_train, y_train)

# Print the best score
print("Best Hyperparameters:")
print(random_search.best_score_)

# Print the best hyperparameters
print("Best Hyperparameters:")
print(random_search.best_params_)


ValueError: Invalid parameter 'rf' for estimator Pipeline(steps=[('pca', PCA()), ('random_forest', RandomForestClassifier())]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].

##Auntomated Hyper Parameter Tuning

###Bayesian Search / Optimization
https://www.analyticsvidhya.com/blog/2021/05/bayesian-optimization-bayes_opt-or-hyperopt/

Bayesian optimization uses probability to find the minimum of a function. The final aim is to find the input value to a function which can gives us the lowest possible output value.It usually performs better than random,grid and manual search providing better performance in the testing phase and reduced optimization time.
In Hyperopt, Bayesian Optimization can be implemented giving 3 three main parameters to the function fmin.

- Objective Function = defines the loss function to minimize.
- Domain Space = defines the range of input values to test (in Bayesian Optimization this space creates a probability distribution for each of the used Hyperparameters).
- Optimization Algorithm = defines the search algorithm to use to select the best input values to use in each new iteration.

In [ ]:
#1:35
# prompt: apply bayesian optimization using hyperopt

from hyperopt import hp, tpe, fmin, STATUS_OK, Trials
from sklearn.model_selection import cross_val_score


def objective(space):
    # Define the hyperparameters
    C = space['C']
    kernel = space['kernel']
    gamma = space['gamma']

    # Create the SVM classifier
    svm_classifier = SVC(C=C, kernel=kernel, gamma=gamma)
    # accuracy = cross_val_score(svm_classifier, X_train, y_train, cv = 5,n_jobs=-1).mean()
    # # Train the classifier on the training data
    svm_classifier.fit(X_train, y_train)

    # # Evaluate the classifier on the validation data
    accuracy = accuracy_score(y_test, svm_classifier.predict(X_test))
    # We aim to maximize accuracy, therefore we return it as a negative value
    # Return the loss (negative accuracy)
    return {'loss': -accuracy, 'status': STATUS_OK, 'model': svm_classifier}


# Define the hyperparameter space
#hp.quniform('max_depth',10,1200,10) integer values
#hp.uniform('min_samples_split', 0,1) float values
space = {
    'C': hp.choice('C', [1, 10, 100, 1000]),
    'kernel': hp.choice('kernel', ['linear', 'rbf']),
    'gamma': hp.choice('gamma', ['scale', 'auto', 0.1, 0.2, 0.3, 0.5, 0.6])
}


# Perform Bayesian optimization
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials)

# Get the best hyperparameters and model
best_params = {key: space[key][best[key]] for key in space}
best_model = trials.best_trial['result']['model']

# Print the results
print("Best Hyperparameters:")
print(best_params)

print("Best Model Accuracy:")
print(accuracy_score(y_test, best_model.predict(X_test)))

In [ ]:
# prompt: train the model with best parameters obtained

# Train the best model with the best hyperparameters
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(classification_rep)


###Optuna
https://www.analyticsvidhya.com/blog/2020/11/hyperparameter-tuning-using-optuna/

In [ ]:
!pip install optuna

In [ ]:
# Import necessary libraries
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Define the objective function
def objective(trial):
    # Define hyperparameters
    C = trial.suggest_float('C', 0.01, 0.9)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    # Create the SVM classifier
    model = SVC(C=C, kernel=kernel, gamma=gamma)

    # Define the cross-validation strategy
    cv = StratifiedKFold(n_splits=3)

    # Evaluate the model using cross-validation
    score = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()

    # Return the negative score as the loss function
    return 1 - score

# Create the Optuna study with a faster sampler (e.g., TPE)
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler())

# Perform hyperparameter tuning with a larger number of trials and fewer concurrent trials
study.optimize(objective, n_trials=20, n_jobs=-1)

# Print the best hyperparameters
print("Best Hyperparameters:")
print(study.best_params)

# Train the best model with the best hyperparameters
best_model = SVC(**study.best_params)
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print('Confusion Matrix:')
print(conf_matrix)
print('Classification Report:')
print(classification_rep)


In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_slice(study)